# Comprehensive Iris Dataset Analysis
This notebook performs exploratory data analysis, statistical hypothesis testing, dimensionality reduction (PCA & LDA), multi-model supervised classification, and unsupervised clustering on the Iris dataset.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, adjusted_rand_score, normalized_mutual_info_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans, AgglomerativeClustering

plt.style.use('ggplot')

## 1. Data Loading & Inspection

In [ ]:
df = pd.read_csv('Iris.csv')
if 'Id' in df.columns:
    df = df.drop(columns=['Id'])

print('Dataset Shape:', df.shape)
print('Missing Values:\n', df.isnull().sum())
print('Duplicate Rows:', df.duplicated().sum())
df.head()

## 2. Descriptive Statistics & Correlations

In [ ]:
feature_cols = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
display(df[feature_cols].describe())

print('\nSpecies-wise Breakdown:')
for sp in df['Species'].unique():
    print(f'=== {sp} ===')
    display(df[df['Species'] == sp][feature_cols].describe())

print('\nCorrelation Matrix:')
corr = df[feature_cols].corr()
display(corr)

## 3. Visualizations

In [ ]:
species_list = df['Species'].unique()
colors = {'Iris-setosa': '#1f77b4', 'Iris-versicolor': '#ff7f0e', 'Iris-virginica': '#2ca02c'}

# Boxplots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, col in enumerate(feature_cols):
    ax = axes[i // 2, i % 2]
    data_to_plot = [df[df['Species'] == sp][col] for sp in species_list]
    bplot = ax.boxplot(data_to_plot, patch_artist=True, tick_labels=species_list)
    for patch, sp in zip(bplot['boxes'], species_list):
        patch.set_facecolor(colors[sp])
    ax.set_title(f'{col} Distribution by Species', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Statistical Testing (ANOVA)

In [ ]:
print('One-Way ANOVA & Kruskal-Wallis Test Results:')
for col in feature_cols:
    groups = [group[col].values for name, group in df.groupby('Species')]
    f_stat, p_val = stats.f_oneway(*groups)
    kw_stat, kw_p = stats.kruskal(*groups)
    print(f'Feature: {col:15s} | ANOVA F = {f_stat:8.2f} (p = {p_val:.2e}) | Kruskal H = {kw_stat:6.2f} (p = {kw_p:.2e})')

## 5. Dimensionality Reduction (PCA & LDA)

In [ ]:
X = df[feature_cols]
y = df['Species']
le = LabelEncoder()
y_encoded = le.fit_transform(y)
X_scaled = StandardScaler().fit_transform(X)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print('PCA Explained Variance Ratio:', pca.explained_variance_ratio_)

# LDA
lda = LDA(n_components=2)
X_lda = lda.fit_transform(X_scaled, y_encoded)
print('LDA Explained Variance Ratio:', lda.explained_variance_ratio_)

## 6. Supervised Machine Learning Benchmark

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'MLP Neural Network': MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=500, random_state=42)
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print(f'{"Model":22s} | {"Test Acc":8s} | {"Precision":9s} | {"Recall":7s} | {"F1 Score":8s} | {"10-Fold CV Mean ± Std"}')
print('-' * 90)
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    cv_scores = cross_val_score(model, X_scaled, y_encoded, cv=cv, scoring='accuracy')
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f'{name:22s} | {acc*100:7.2f}% | {prec:9.4f} | {rec:7.4f} | {f1:8.4f} | {cv_scores.mean()*100:6.2f}% ± {cv_scores.std()*100:4.2f}%')

## 7. Unsupervised Clustering Evaluation

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
km_labels = kmeans.fit_predict(X_scaled)

agg = AgglomerativeClustering(n_clusters=3)
agg_labels = agg.fit_predict(X_scaled)

print('K-Means ARI:', adjusted_rand_score(y_encoded, km_labels), '| NMI:', normalized_mutual_info_score(y_encoded, km_labels))
print('Agglomerative ARI:', adjusted_rand_score(y_encoded, agg_labels), '| NMI:', normalized_mutual_info_score(y_encoded, agg_labels))